## This one is for making a systemC netlist of synthesized circuits, combinational and sequential

In [14]:
import os

In [15]:
benchmark_name = "s5378_netlist_scanInserted"

In [16]:
## verilog preprocess:
file_path = benchmark_name+".v"
preVer = []
with open(file_path, "r") as file:
    for line in file:
        if "not" in line:
            outp = (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" ")
            inp = (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" ")
            line = line.replace("not ", "nand ").replace("NOT1_","NAND2_rep").replace(inp+");", inp+", "+inp+");")
            preVer.append(line)
        else:
            preVer.append(line)

directory_path = benchmark_name+"/Verilog/"
os.makedirs(directory_path, exist_ok=True)
Ver_path = os.path.join(directory_path+benchmark_name+".v")
with open(Ver_path, "w") as file:
    file.writelines(preVer)
file.close()


In [17]:
## One input gates
class gate1:
    def __init__(self, name, numOfInp, outputZN, inputA1, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.outputZN = outputZN
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n ZN = {self.outputZN}\n gate Type = {self.type}"


In [18]:
## Two input gates
class gate2:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.outputZN = outputZN
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n ZN = {self.outputZN}\n gate Type = {self.type}"


In [19]:
## Three input gates
class gate3:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, gateNum):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n ZN = {self.outputZN}"


In [20]:
## Four input gates
class gate4:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, inputA4, gateNum):
        self.name = name
        self.gateNum = gateNum
        self.numOfInp = numOfInp
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.inputA4 = inputA4
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n A4 = {self.inputA4}\n ZN = {self.outputZN}"


In [45]:
## Four input gates
class DFF:
    def __init__(self, name, numOfInp, clk, CE, CLR, D, NbarT, PRE, Q, Si, global_reset, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.clk = clk
        self.Q = Q
        self.D = D
        self.CE = CE
        self.CLR = CLR
        self.NbarT = NbarT
        self.PRE = PRE
        self.Si = Si
        self.global_reset = global_reset
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n D = {self.D}\n Q = {self.Q}\n gate Type = {self.type}"


In [48]:
directory_path = benchmark_name+"/Verilog/"
file_path = os.path.join(directory_path+benchmark_name+".v")
i = 0
j = 0
net = []
required_gates = []
newGate = False
PIs = []
POs = []
assign_left = []
assign_right = []

numOfPI = 0
numOfPO = 0

wires=[]
wire_lines=""
with open(file_path, "r") as file:
    for line in file:
        if 'wire' in line:
            if(';' in line):
                wire_lines += line.rstrip('\n').split('wire')[1]
            else:
                wire_lines += line.rstrip('\n').split('wire')[1]
                while not(';' in line):
                    line = file.readline()  
                    wire_lines += line.rstrip('\n')
        wires = wire_lines
wires = wires.replace(" ", "").replace(";",",").split(',')
if (wires[len(wires)-1] == ""):
    wires = wires[:-1]
else: 
    wires[len(wires)-1] = wires[len(wires)-1].strip(',')
numOfwire = len(wires)
file.close()


with open(file_path, "r") as file:
    for line in file:
        if 'input' in line:
            PI= line.rstrip('\n').split('input')[1].strip(';').replace(" ", "")
            numOfPI += 1
            # print(numOfPI)
            PIs.append(PI)
            # PI_lines = line.rstrip('\n').split('input')[1]
            # ## handling enters
            # while ';' not in line:
            #     line = file.readline()
            # numOfPI = len(PI_lines.split()[1:])
            # PIs = PI_lines.replace(" ", "").strip(';').split(',')

        elif 'output' in line:
            PO= line.rstrip('\n').split('output')[1].strip(';').replace(" ", "")
            numOfPO += 1
            # print(numOfPO)
            POs.append(PO)
            # PO_lines = line.rstrip('\n').split('output')[1]
            # while ';' not in line:
            #     line = file.readline()
            #     PO_lines += line.rstrip('\n')
            # numOfPO = len(PO_lines.split()[1:])
            # POs = PO_lines.replace(" ", "").strip(';').split(',')

        elif 'dff' in line:
            # gateInps =[]
            print(line)
            newGate = True
            numOfGateInput = -1 # meand DFF
            j=j+1
            currGateName = line.split()[1].strip("(")
            print(currGateName)
            # print(line.split()[1][0:5])
            if ("DFF" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("DFF")
            line = file.readline() ##first input
            C = (line.split('(')[1].split(')')[0])
            print ("C : "+str(C))

            line = file.readline()
            CE = (line.split('(')[1].split(')')[0])
            print ("CE : "+str(CE))

            line = file.readline()
            CLR = (line.split('(')[1].split(')')[0])

            line = file.readline() ##first input
            D = (line.split('(')[1].split(')')[0])
            print ("D : "+str(D))

            line = file.readline()
            NbarT = (line.split('(')[1].split(')')[0])

            line = file.readline()
            PRE = (line.split('(')[1].split(')')[0])

            line = file.readline() ##first input
            Q = (line.split('(')[1].split(')')[0])
            print ("Q : "+str(Q))

            line = file.readline()
            Si = (line.split('(')[1].split(')')[0])

            line = file.readline()
            global_reset = (line.split('(')[1].split(')')[0])
            
            gateType = "dff"
            
        
        elif 'NAND' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1][0:5]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("NAND" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("NAND")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "NAND"+str(numOfGateInput)+"_X1"
            # if (numOfGateInput == 1):
            #     net.append(gate1(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 2):
            #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 3):
            #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'NOR' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1][0:5]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("NOR" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("NOR")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "NOR"+str(numOfGateInput)+"_X1"
            # newGate = True
            # i=i+1
            # currGateName = line.split()[1][0:4]
            # if (line.split()[1][0:4] not in required_gates):
            #     required_gates.append(line.split()[1][0:4])
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            # # if (numOfGateInput == 2):
            # #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # # if (numOfGateInput == 3):
            # #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'NOT' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1][0:5]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("INV" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("INV")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "INV"+str(numOfGateInput)+"_X1"
            # newGate = True
            # i=i+1
            # currGateName = line.split()[1][0:4]
            # currGateName = "INV"
            # if (currGateName not in required_gates):
            #     required_gates.append(currGateName)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1

        elif 'assign' in line:
            # print(line.split())
            assign_left.append(line.split()[1])
            assign_right.append(line.split()[3].strip(';'))

        # elif ' or' in line:
        #     # print(line)
        #     newGate = True
        #     i=i+1
        #     numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
        #     currGateName = "OR"+str(numOfGateInput)
        #     # currGateName = line.split()[1][0:3]
        #     if (currGateName not in required_gates):
        #         required_gates.append(currGateName)
        #     # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1


        # elif 'and' in line:
        #     newGate = True
        #     i=i+1
        #     numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
        #     currGateName = "AND"+str(numOfGateInput)
        #     if (currGateName not in required_gates):
        #         required_gates.append(currGateName)
        else:
            newGate = False

        if (newGate):
            if (numOfGateInput == 1):
                # net.append(gate1(currGateName, 1, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), i))
                net.append(gate1(currGateName, 1, gateOutp, gateInps[0], i, gateType))

            elif (numOfGateInput == 2):
                # net.append(gate2(currGateName, 2,(line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
                net.append(gate2(currGateName, 2, gateOutp, gateInps[0], gateInps[1], i, gateType))

            elif (numOfGateInput == 3):
                net.append(gate3(currGateName, 3, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
            elif (numOfGateInput == 4):
                net.append(gate4(currGateName, 4, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[4].strip(" "), i))
            elif (numOfGateInput == -1): # DFF
                net.append(DFF(currGateName, -1, C, CE, CLR, D, NbarT, PRE, Q, Si, global_reset, j, gateType))
file.close()

 dff DFF_0(

DFF_0
C : CK
CE : 1'b1
D : n2897gat
Q : n672gat
 dff DFF_1(

DFF_1
C : CK
CE : 1'b1
D : n3069gat
Q : II729
 dff DFF_10(

DFF_10
C : CK
CE : 1'b1
D : n3065gat
Q : II368
 dff DFF_100(

DFF_100
C : CK
CE : 1'b1
D : n3053gat
Q : II2425
 dff DFF_101(

DFF_101
C : CK
CE : 1'b1
D : n2613gat
Q : n2506gat
 dff DFF_102(

DFF_102
C : CK
CE : 1'b1
D : n1625gat
Q : n1834gat
 dff DFF_103(

DFF_103
C : CK
CE : 1'b1
D : n1626gat
Q : n1767gat
 dff DFF_104(

DFF_104
C : CK
CE : 1'b1
D : n1442gat
Q : n2084gat
 dff DFF_105(

DFF_105
C : CK
CE : 1'b1
D : n2482gat
Q : n1787gat
 dff DFF_106(

DFF_106
C : CK
CE : 1'b1
D : n2557gat
Q : n1785gat
 dff DFF_107(

DFF_107
C : CK
CE : 1'b1
D : n2487gat
Q : n1633gat
 dff DFF_108(

DFF_108
C : CK
CE : 1'b1
D : n2532gat
Q : n1884gat
 dff DFF_109(

DFF_109
C : CK
CE : 1'b1
D : n2486gat
Q : n1631gat
 dff DFF_11(

DFF_11
C : CK
CE : 1'b1
D : n3067gat
Q : II359
 dff DFF_110(

DFF_110
C : CK
CE : 1'b1
D : n2353gat
Q : n1989gat
 dff DFF_111(

DFF_111
C : CK
CE :

In [49]:
print(len(assign_left))
print(len(assign_right))


980
980


In [50]:
print(net[5])

 gateNum = 6
 gateName = _2423
 A1 = _0880_
 ZN = _0975_
 gate Type = INV1_X1


In [51]:
# num of gates
len(net)

1505

In [52]:
required_gates

['INV', 'NOR', 'NAND', 'DFF']

In [53]:
print(len(PIs))
print(len(POs))

39
50


In [54]:
print(net[1328])

 gateNum = 3
 gateName = DFF_10
 D = n3065gat
 Q = II368
 gate Type = dff


In [61]:
## systemC output
sysC_H = []
# for required_gate in required_gates:
#     sysC_H.append("#include \""+required_gate+"_X1.h\"\n")
sysC_H.append("#include \"Complex_NAgate_45.h\"\n")

sysC_H.append("SC_MODULE("+benchmark_name+")\n{\n")

sysC_H.append("\tsc_in <sc_logic> clk;\n")
sysC_H.append("\tsc_in <sc_logic> rst;\n")
sysC_H.append("\tsc_in <sc_logic> PbarS;\n")
sysC_H.append("\tsc_in <sc_logic> Si;\n")
sysC_H.append("\tsc_out <sc_logic> So;\n\n")
sysC_H.append("\tsc_signal <sc_logic> sc_logic_1_signal, sc_logic_0_signal;\n\n")


sysC_H.append("\tsc_in <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_H.append(PIs[PI_idx]+", ")
sysC_H.append(PIs[len(PIs)-1]+";\n")

sysC_H.append("\tsc_out <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_H.append(POs[PO_idx]+", ")
sysC_H.append(POs[len(POs)-1]+";\n")

sysC_H.append("\tsc_signal <sc_logic> ")
for wire_idx in range(len(wires)-1):
    if((not(wires[wire_idx] in PIs)) and (not(wires[wire_idx] in POs))):
        sysC_H.append(wires[wire_idx]+", ")
sysC_H.append(wires[len(wires)-1]+";\n")

sysC_H.append("\tsc_in<sc_logic> endSim; \n")
sysC_H.append("\tsc_in<sc_logic> newTV; \n")
sysC_H.append("\tsc_uint<32> counter; \n")

sysC_H.append("\n\tint numOfGates;\n\tint totalObservedCombs;\n\tdouble GIC_Coverage;\n\n")


# sysC_H.append("\n\tint numOfGates;\n\tdouble t;\n\tdouble outLoad;\n\n")

i = 1
for g in net:
    # sysC_H.append("\t"+g.name+"_X1* "+g.name+"_Gate"+str(i)+";\n")
    sysC_H.append("\t"+g.type+"* "+g.name+"_Gate"+str(i)+";\n")
    i = i+1

sysC_H.append("\n\tSC_CTOR("+benchmark_name+")\n\t{\n\t\tnumOfGates = "+str(len(net))+";\n\n")

i = 1
for g in net:
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.name+"_X1(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.type+"(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    if (g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->C"+"("+g.clk+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->CE"+"(sc_logic_1_signal);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->CLR"+"("+g.CLR+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->D"+"("+g.D+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->NbarT"+"("+g.NbarT+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->PRE"+"(sc_logic_0_signal);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->Si"+"("+g.Si+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->global_reset"+"(sc_logic_0_signal);\n")

    if (g.numOfInp == 1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")

    if (g.numOfInp == 2):
        if (g.name.startswith("XOR")): ## XOR has only 2
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->B"+"("+g.inputA2+");\n")
        else:    
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
    if (g.numOfInp == 3):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
    if (g.numOfInp == 4):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A4"+"("+g.inputA4+");\n")
    if(g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->Q"+"("+g.Q+");\n\n")
    else:
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->ZN"+"("+g.outputZN+");\n\n")
    # if (g.outputZN in POs):
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c"+" = outLoad;\n")
    # else:
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c = 0;\n")
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->aged_time = t;\n\n")
    i = i+1

sysC_H.append("\t\tcout << \"all gates are instantiated \" << numOfGates << \"\\n\";\n")

sysC_H.append("\t\tSC_METHOD(sc_logic_signal_assignments);\n")

sysC_H.append("\t\tSC_THREAD(assignments);\n")
# //sensitivity
j=0
unique_sens_list = list(set(assign_right))
sysC_H.append("\t\tsensitive")
for j in range(len(unique_sens_list)):
    sysC_H.append(" << "+unique_sens_list[j])
sysC_H.append(";\n")
sysC_H.append("\t\tSC_METHOD(GIC_Coverage_Calculator);\n")
sysC_H.append("\t\tsensitive << endSim<< newTV;\n\n\t}\n")

sysC_H.append("\tvoid ini();\n\tvoid assignments();\n\tvoid GIC_Coverage_Calculator();\n")

sysC_H.append("\tvoid sc_logic_signal_assignments(){\n\t\tsc_logic_1_signal.write(SC_LOGIC_1);\n\t\tsc_logic_0_signal.write(SC_LOGIC_0);\n\t}\n")


# j = 1
# for required_gate_idx in range(len(required_gates)):
#     sysC_H.append("\tvoid notifyAlfaCalc"+required_gates[required_gate_idx]+"(")
#     sysC_H.append(required_gates[required_gate_idx]+"_X1 *gate"+");\n")
#     # sysC_H.append(required_gates[len(required_gates)-1]+"_X1 *gate"+str(j)+");\n};\n")
#     j=j+1
sysC_H.append("\n};\n")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"netlist.h")
with open(sysC_output_file_path, "w") as file:
    file.writelines(sysC_H)
file.close()

In [62]:
sysC_C = []
sysC_C.append("#include \"netlist.h\"\n#include <cmath>\n")
sysC_C.append("std::ofstream GIC_logFile(\"GIC_logFile.txt\");\n\n")
sysC_C.append("void "+benchmark_name+"::assignments()\n{\n\twhile (true)\n\t{\n")

m=0
for m in range(len(assign_left)):
    sysC_C.append("\t\t"+assign_left[m]+".write("+assign_right[m]+");\n")

sysC_C.append("\n\t\twait();\n\t}\n}\n\n")

sysC_C.append("void "+benchmark_name+"::GIC_Coverage_Calculator()\n{\n")
sysC_C.append("\ttotalObservedCombs =\n")
n=0
for n in range(len(net)):
    if n==len(net)-1:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs;\n")
    else:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs +\n")

sysC_C.append("\tcout << \"END OF SIM=> Total Observed Combs = \" << totalObservedCombs << \"\\n\";\n")
sysC_C.append("\tcout << \"GIC Coverage = \" << GIC_Coverage << \"\\n\";\n\n}")



directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_C_output_file_path = os.path.join(directory_path+"netlist.cpp")
with open(sysC_C_output_file_path, "w") as file:
    file.writelines(sysC_C)
file.close()

In [63]:

sysC_TB_H = []
sysC_TB_H.append("#include \"netlist.h\"\n#include <fstream>\n\nSC_MODULE("+benchmark_name+"_TB)\n{\n\n")
sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_TB_H.append("testData"+str(PI_idx+1)+", ")
sysC_TB_H.append("testData"+str(len(PIs))+";\n")

sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_TB_H.append("testRes"+str(PO_idx+1)+", ")
sysC_TB_H.append("testRes"+str(len(POs))+";\n")

sysC_TB_H.append("\n\tsc_signal<sc_logic> reset, clock, end, newTV;\n\t"+benchmark_name+"* UUT;\n\n")
sysC_TB_H.append("\tstd::vector<std::string> testVecs;\n")

sysC_TB_H.append("\tSC_CTOR("+benchmark_name+"_TB)\n\t{\n\n\t\ttestVecs = read_testPtr (\"testPatterns.txt\");\n\t\tUUT = new "+benchmark_name+"(\""+benchmark_name+"_instance\");\n")
p=1
for PI in PIs:
    sysC_TB_H.append("\t\tUUT->"+PI.strip(" ")+"(testData"+str(p)+");\n")
    p=p+1

o=1
for PO in POs:
    sysC_TB_H.append("\t\tUUT->"+PO.strip(" ")+"(testRes"+str(o)+");\n")
    o=o+1

sysC_TB_H.append("\t\tUUT->endSim(end);\n")
sysC_TB_H.append("\t\tUUT->newTV(newTV);\n")
sysC_TB_H.append("\n\t\tSC_THREAD(testPtr);\n\t\tSC_THREAD(endOfSim);\n\t}\n")

sysC_TB_H.append("\n\tvoid endOfSim();\n\tvoid testPtr();\n\tstd::vector<std::string> read_testPtr(std::string filename);\n")
sysC_TB_H.append("};\n")




directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_H_output_file_path = os.path.join(directory_path+"TB.h")
with open(sysC_TB_H_output_file_path, "w") as file:
    file.writelines(sysC_TB_H)
file.close()

In [64]:
sysC_TB_C = []
sysC_TB_C.append("#include \"TB.h\"\n\n")
sysC_TB_C.append("void "+benchmark_name+"_TB::testPtr()\n{\n\twhile (true)\n\t{\n")

m=1
for PI in PIs:
    # sysC_TB_C.append("void powerGatesNetlistTB::testData"+str(m)+"Waveform()\n{\n\twhile (true)\n{\n")
    sysC_TB_C.append("\t\ttestData"+str(m)+".write(SC_LOGIC_0);\n")
    m=m+1

sysC_TB_C.append("\t\twait(1000, SC_NS);\n")
sysC_TB_C.append("\t\tfor (int testVecidx = 0; testVecidx < testVecs.size(); testVecidx++)\n\t\t\t{\n")
s=0
for s in range(len(PIs)):
    sysC_TB_C.append("\t\t\tif (testVecs[testVecidx]["+str(s)+"] == '0')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_0);\n\t\t\telse if (testVecs[testVecidx]["+str(s)+"] == '1')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_1);\n\n")
sysC_TB_C.append("\t\twait(15000, SC_NS);\n\t\tnewTV.write(SC_LOGIC_1);\n\t\twait(0, SC_NS);\n\t\t}\n\t\twait();\n\t}\n}\n")

sysC_TB_C.append("std::vector<std::string> "+benchmark_name+"_TB::read_testPtr(std::string filename)\n{\n")
sysC_TB_C.append("\tstd::vector<std::string> selected_testVec;\n\tstd::ifstream file(filename);\n\tif (!file.is_open()) {\n\t\tstd::cerr << \"Error opening file!\" << std::endl;\t\n\t}\n\tstd::string line;\n\tstd::vector<std::string> lines;\n\twhile (std::getline (file, line))\n\t{\n\t\tlines.push_back(line);\n\t}\n\treturn lines;\n}")

sysC_TB_C.append("\nvoid "+benchmark_name+"_TB::endOfSim()\n{\n\twhile (true)\n{\n\t\tend.write(SC_LOGIC_0);\n\t\twait(3900000, SC_NS);\n\t\tend.write(SC_LOGIC_1);\n\t\twait(15000, SC_NS);\n\t\twait();\n\t}\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_C_output_file_path = os.path.join(directory_path+"TB.cpp") 
with open(sysC_TB_C_output_file_path, "w") as file:
    file.writelines(sysC_TB_C)
file.close()

In [65]:
sysC_sim_C = []
sysC_sim_C.append("#include \"TB.h\"\n#include <iostream>\n#include <fstream> \n\nint sc_main(int argc, char** argv)\n{\n\t"+benchmark_name+"_TB* TOP = new "+benchmark_name+"_TB(\"netlistSimulationTB_instance\");")
sysC_sim_C.append("\n\tsc_start(400000, SC_NS);\n\treturn 0;\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_sim_C_output_file_path = os.path.join(directory_path+"simulation.cpp") 
with open(sysC_sim_C_output_file_path, "w") as file:
    file.writelines(sysC_sim_C)
file.close()